## **1. SETUP LIBRARY dan DATASET**

Sebelum membangun arsitektur Transformer, langkah pertama adalah mengimpor (*import*) berbagai library yang diperlukan. Library tersebut digunakan untuk mendukung proses pemrosesan data, pembangunan model, pelatihan, hingga evaluasi.

Selanjutnya, dataset diambil dari **Hugging Face Datasets** menggunakan pustaka `datasets`. Untuk mengakses beberapa dataset yang memiliki batasan akses atau memerlukan autentikasi, disarankan untuk melakukan login ke akun Hugging Face terlebih dahulu. Dengan demikian, proses pengambilan dataset dapat berjalan dengan lebih lancar tanpa mengalami kendala perizinan (*authentication*).


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from datasets import load_dataset
from huggingface_hub import login
import json
import matplotlib.pyplot as plt
import math

In [3]:
# Gw login dahulu agar bisa mendapatkan akses dataset untuk language model
login()

In [4]:
# Melakukan pengambilan dataset
# Biasanya menunggu 26-30 menit
dataset_llm = load_dataset("nampdn-ai/tiny-strange-textbooks")

In [5]:
# Saya membuat fungsi untuk menapilkan datasetnya
# Ini buat mengecek, apakah sudah berhasil atau belum.
def show_sample(index):
    sample = dataset_llm["train"][index]

    print("=" * 100)
    print(f"DATASET #{index}")
    print("=" * 100)

    print(sample["text"])

    print("=" * 100)

show_sample(0)

DATASET #0
 Title: The Significance of the Past in Shaping the Future

Introduction:
The past plays a significant role in shaping the future, especially for the people of Israel. This textbook will explore the significance of the past in shaping the future for the Jewish people, focusing on the example of Ezra's exodus.

Chapter 1: The Importance of the Past
The past is an essential part of a people's identity and heritage. For the Jewish people, their history is rich with stories of struggle, survival, and triumph. These stories serve as a source of pride and inspiration for the present generation.

Section 1: The Significance of the Exodus
The Exodus from Egypt is one of the most significant events in Jewish history. It represents the liberation of the Jewish people from slavery and their journey towards freedom. The story of the Exodus has been passed down through generations and serves as a reminder of the resilience and strength of the Jewish people.

Section 2: Ezra's Exodus
Ezra

## **2. PEMBAGIAN dan TOKENISASI DATA**

Pada tahap ini, dataset dibagi menjadi tiga bagian, yaitu **training**, **validation**, dan **test**. Pembagian ini bertujuan agar model dapat dilatih, dievaluasi selama proses pelatihan, serta diuji menggunakan data yang belum pernah dilihat sebelumnya.

Dataset **training** digunakan untuk melatih model agar mampu mempelajari pola dari data. Dataset **validation** digunakan untuk mengevaluasi performa model pada setiap iterasi atau *epoch* pelatihan, sehingga dapat membantu memantau proses pembelajaran dan mencegah *overfitting*. Sementara itu, dataset **test** digunakan sebagai evaluasi akhir untuk mengukur kemampuan model dalam melakukan generalisasi terhadap data baru.

Selain itu, dilakukan proses **tokenisasi**, yaitu mengubah teks menjadi serangkaian token yang kemudian dipetakan menjadi **token ID**. Representasi numerik ini memungkinkan model Transformer memproses teks secara efisien sebagai masukan sebelum dikonversi menjadi embedding.


In [6]:
# Melakukan pembagian data antara validasi, test, dan train
# Ini berguna untuk evaluasinya agar lebih jelas dan meyakinkan
# 98% train, 2% sementara
splits = dataset_llm["train"].train_test_split(
    test_size=0.02,
    seed=42
)

train_dataset = splits["train"]
temp_dataset = splits["test"]

# 1% validation, 1% test
temp_splits = temp_dataset.train_test_split(
    test_size=0.5,
    seed=42
)

validation_dataset = temp_splits["train"]
test_dataset = temp_splits["test"]

print(train_dataset)
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['text'],
    num_rows: 2689685
})
Dataset({
    features: ['text'],
    num_rows: 27446
})
Dataset({
    features: ['text'],
    num_rows: 27446
})


In [7]:
# Melakukan proses tokenisasi
# Transfomers butuh pemisahan antar karakter, jika tidak dibuatkan tokenisasi akan susah membedakan input tersebut
# Ini proses pemecahan dan mengubah menjadi ID tiap token

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Membuat fungsi agar bisa menerima objek Dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512
    )

# Di sini, saya memberikan atau menampilkan cara implementasinya
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

# Lalu cek hasil tokenisasinya
# Gw hanya menampilkan bagian training saja
print(tokenized_train["input_ids"])

Column([[101, 2516, 1024, 6209, 18918, 1999, 2563, 4955, 1024, 2563, 2003, 1037, 3376, 2406, 2007, 14726, 12793, 1010, 5291, 4564, 1010, 1998, 23273, 10833, 1012, 1037, 6209, 9151, 1999, 2563, 2003, 1996, 3819, 2126, 2000, 8849, 2023, 12916, 2406, 1998, 3325, 1996, 2190, 1997, 2329, 3226, 1012, 1999, 2023, 16432, 1010, 2057, 2097, 4553, 2055, 1996, 2367, 4127, 1997, 18918, 2800, 1999, 2563, 1010, 1996, 17363, 1010, 1998, 1996, 2367, 4655, 2000, 3942, 1012, 2057, 2097, 2036, 4553, 2055, 1996, 2367, 5269, 2073, 6209, 18918, 2024, 2800, 1010, 1998, 2129, 2000, 5454, 1996, 2157, 9151, 2005, 2115, 3791, 1012, 3127, 1015, 1024, 4127, 1997, 18918, 1999, 2563, 1999, 2023, 3127, 1010, 2057, 2097, 4553, 2055, 1996, 2367, 4127, 1997, 6209, 18918, 2800, 1999, 2563, 1012, 2122, 2421, 1024, 1015, 1012, 2406, 18918, 1024, 2122, 2024, 3151, 18918, 2284, 1999, 3541, 2752, 1010, 5129, 2011, 10833, 1998, 16439, 1012, 2027, 2024, 3819, 2005, 2216, 2040, 2215, 2000, 3325, 1996, 3521, 1998, 25283, 26147, 30

In [8]:
print(tokenized_train)

Dataset({
    features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2689685
})


In [9]:
# Gw akan ubah menjadi Tensor pytorch agar bisa diterima oleh libary deep learning tersebut
tokenized_train.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

tokenized_validation.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

tokenized_test.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

# Gw hanya memberikan atau menampilkan training saja
print(tokenized_train[0])

{'input_ids': tensor([  101,  2516,  1024,  6209, 18918,  1999,  2563,  4955,  1024,  2563,
         2003,  1037,  3376,  2406,  2007, 14726, 12793,  1010,  5291,  4564,
         1010,  1998, 23273, 10833,  1012,  1037,  6209,  9151,  1999,  2563,
         2003,  1996,  3819,  2126,  2000,  8849,  2023, 12916,  2406,  1998,
         3325,  1996,  2190,  1997,  2329,  3226,  1012,  1999,  2023, 16432,
         1010,  2057,  2097,  4553,  2055,  1996,  2367,  4127,  1997, 18918,
         2800,  1999,  2563,  1010,  1996, 17363,  1010,  1998,  1996,  2367,
         4655,  2000,  3942,  1012,  2057,  2097,  2036,  4553,  2055,  1996,
         2367,  5269,  2073,  6209, 18918,  2024,  2800,  1010,  1998,  2129,
         2000,  5454,  1996,  2157,  9151,  2005,  2115,  3791,  1012,  3127,
         1015,  1024,  4127,  1997, 18918,  1999,  2563,  1999,  2023,  3127,
         1010,  2057,  2097,  4553,  2055,  1996,  2367,  4127,  1997,  6209,
        18918,  2800,  1999,  2563,  1012,  2122, 

## **3. EMBEDDING dan POSITIONAL ENCODING (RoPE)**

**Embedding** merupakan proses mengubah setiap **token ID** menjadi sebuah vektor berdimensi tetap (*dense vector*). Vektor ini berisi representasi numerik yang dipelajari selama proses pelatihan sehingga token-token yang memiliki makna atau konteks yang mirip akan memiliki representasi vektor yang berdekatan di ruang embedding. Representasi ini menjadi masukan utama bagi arsitektur Transformer.

Namun, mekanisme *self-attention* pada Transformer tidak memiliki informasi mengenai urutan token karena setiap token diproses secara paralel. Oleh karena itu, dibutuhkan mekanisme untuk menyisipkan informasi posisi ke dalam proses attention.

Berbeda dengan *Positional Encoding* sinusoidal klasik yang **dijumlahkan langsung ke embedding token** sebelum masuk ke Transformer, **Rotary Positional Embedding (RoPE)** menyisipkan informasi posisi dengan cara **merotasi vektor Query ($Q$) dan Key ($K$) di dalam setiap layer attention**, bukan pada tahap embedding awal. Dengan pendekatan ini, informasi posisi bersifat *relatif* antar token (bergantung pada selisih posisi $m-n$), bukan absolut, sehingga RoPE cenderung lebih baik dalam menggeneralisasi ke panjang urutan (*sequence length*) yang lebih panjang daripada saat pelatihan.

### Cara Kerja RoPE

Untuk setiap pasangan dimensi $(2i, 2i+1)$ pada vektor $Q$ atau $K$ di posisi $pos$, RoPE menerapkan rotasi menggunakan matriks rotasi 2 dimensi:

$$
\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix}
=
\begin{pmatrix} \cos(pos \cdot \theta_i) & -\sin(pos \cdot \theta_i) \\ \sin(pos \cdot \theta_i) & \cos(pos \cdot \theta_i) \end{pmatrix}
\begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}
$$

dengan frekuensi rotasi:

$$
\theta_i = 10000^{-\frac{2i}{d_{\mathrm{model}}}}
$$

dengan:
* $pos$ adalah posisi token dalam urutan.
* $i$ adalah indeks pasangan dimensi embedding ($i = 0, 1, \dots, d_{\mathrm{model}}/2 - 1$).
* $d_{\mathrm{model}}$ adalah jumlah dimensi embedding.
* $q_{2i}, q_{2i+1}$ adalah pasangan elemen vektor $Q$ (berlaku sama untuk $K$) sebelum rotasi.

Rotasi ini diterapkan pada $Q$ dan $K$ **setelah** proyeksi linear, tepat sebelum perhitungan skor attention. Sifat kunci dari rotasi ini: hasil *dot product* antara $Q$ di posisi $m$ dan $K$ di posisi $n$ hanya bergantung pada selisih relatif $(m-n)$, bukan posisi absolut masing-masing:

$$
\langle f_q(x_m, m), f_k(x_n, n) \rangle = g(x_m, x_n, m-n)
$$

Sehingga alur masukan Transformer menjadi:

$$
\mathbf{X} = \mathbf{E}
$$

$$
Q' = \mathrm{RoPE}(Q, pos), \quad K' = \mathrm{RoPE}(K, pos)
$$

dengan:
* $\mathbf{E}$ adalah matriks embedding token (tanpa penjumlahan PE).
* $Q, K$ adalah hasil proyeksi linear dari $\mathbf{X}$.
* $Q', K'$ adalah $Q$ dan $K$ setelah dirotasi berdasarkan posisi, yang kemudian digunakan dalam perhitungan attention.

In [10]:
# Embedding mengubah setiap token ID menjadi vektor berdimensi d_model.
# Nilai vektor ini dipelajari selama training sehingga token yang sering
# muncul dalam konteks yang mirip akan memiliki representasi yang mirip

# Set parameter
vocab_size = tokenizer.vocab_size
d_model = 512

# Implementasi penggunaan fungsi embedding
embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model
)

# Ini implementasi embedding terhadap training, validation, dan test 
input_ids_train = tokenized_train[0]["input_ids"]
input_ids_test = tokenized_test[0]["input_ids"]
input_ids_validation = tokenized_validation[0]["input_ids"]

embedded_train = embedding(input_ids_train)
embedded_validation = embedding(input_ids_validation)
embedded_test = embedding(input_ids_test)

# Mengecek hasil dari embedding
# DI sini, gw hanya mengecek bentuknya saja
print(embedded_train.shape)
print(embedded_test.shape)
print(embedded_validation.shape)

# Lebih jelas lagi, gw akan membuat output seperti ini
# CUkup 10 sample saja
print(embedded_train[:10])

torch.Size([512, 512])
torch.Size([512, 512])
torch.Size([512, 512])
tensor([[ 0.0109,  0.0370, -0.3328,  ..., -1.8966,  0.6284, -0.6955],
        [-0.8938, -0.8774,  1.0844,  ...,  0.7000,  0.7415, -1.2700],
        [ 0.9028,  1.0483, -0.9286,  ...,  0.1182,  0.9828,  1.9100],
        ...,
        [-1.0599, -0.0199,  1.6313,  ..., -0.4475, -0.7091,  0.1713],
        [ 0.9028,  1.0483, -0.9286,  ...,  0.1182,  0.9828,  1.9100],
        [-1.1180,  0.0524,  0.4589,  ..., -0.1120, -1.6332, -1.0221]],
       grad_fn=<SliceBackward0>)


In [12]:
class RotaryPositionEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)  # [max_seq_len, dim/2]

        self.register_buffer("cos", torch.cos(freqs))
        self.register_buffer("sin", torch.sin(freqs))

    def forward(self, x):
        """
        x: [batch, seq_len, dim]  (khusus MLA, kita selalu pakai bentuk 3 dimensi)
        """
        seq_len = x.size(1)
        cos = self.cos[:seq_len].unsqueeze(0)  # [1, seq_len, dim/2]
        sin = self.sin[:seq_len].unsqueeze(0)

        x1 = x[..., 0::2]
        x2 = x[..., 1::2]

        out = torch.zeros_like(x)
        out[..., 0::2] = x1 * cos - x2 * sin
        out[..., 1::2] = x1 * sin + x2 * cos
        return out

# Implementasi pada embedding
# Gw perlu set panjang urutannya
seq_len = embedded_train.shape[0]
embed_dim = embedded_train.shape[1]

# Membuat layer Positional Encoding
pos_encoder = RotaryPositionEmbedding(
    seq_len,
    embed_dim
)

embedded_train = pos_encoder(embedded_train)
embedded_validation = pos_encoder(embedded_validation)
embedded_test = pos_encoder(embedded_test)

# Menampilkan hasilnya
print(embedded_train.shape)
print(embedded_train[:5])

print(embedded_validation.shape)
print(embedded_validation[:5])

torch.Size([512, 512])
tensor([[ 0.0109,  0.0370, -0.3328,  ..., -1.8966,  0.6284, -0.6955],
        [ 1.1698, -0.4476,  1.4426,  ...,  0.7000,  0.7417, -1.2699],
        [ 0.2033, -1.3684,  0.0198,  ...,  0.1184,  0.9820,  1.9105],
        [-0.0476, -0.9288, -1.0715,  ..., -0.1250, -0.2807, -0.2359],
        [-0.5600,  0.7523, -0.0594,  ...,  0.1343, -1.7501, -1.9785]],
       grad_fn=<SliceBackward0>)
torch.Size([512, 512])
tensor([[ 0.0109,  0.0370, -0.3328,  ..., -1.8966,  0.6284, -0.6955],
        [ 1.4735, -0.6232,  0.9301,  ...,  1.7160, -0.0294, -1.1624],
        [-1.5088,  0.8177, -0.8920,  ..., -0.0118, -0.2944, -0.7475],
        [ 1.4710, -0.5008, -0.4899,  ..., -1.0468, -0.1573,  0.2815],
        [ 0.9981, -0.7566,  2.0765,  ...,  0.6999,  0.7425, -1.2694]],
       grad_fn=<SliceBackward0>)


## **4. MULTI-HEAD LATENT ATTENTION (MLA)**

**Multi-Head Latent Attention (MLA)** adalah varian mekanisme attention yang diperkenalkan pada arsitektur DeepSeek-V2 untuk mengatasi salah satu bottleneck utama pada inference Transformer skala besar, yaitu **ukuran KV cache**. Pada Multi-Head Attention (MHA) standar, setiap token harus menyimpan vektor **Key** dan **Value** penuh untuk setiap head, sehingga ukuran cache tumbuh linear terhadap jumlah head ($n_h$) dan dimensi per head ($d_h$). MLA mengatasi ini dengan mengompresi Key dan Value ke dalam sebuah **vektor laten berdimensi rendah** sebelum disimpan, sehingga hanya vektor laten inilah yang perlu di-cache, bukan $K$ dan $V$ untuk seluruh head.

### 4.1 Kompresi Low-Rank untuk Key dan Value

Alih-alih memproyeksikan hidden state $\mathbf{h}_t \in \mathbb{R}^{d_{\mathrm{model}}}$ langsung menjadi $K$ dan $V$ untuk setiap head, MLA terlebih dahulu memproyeksikannya ke sebuah **vektor laten bersama (joint compressed latent)** berdimensi kecil $d_c \ll n_h \cdot d_h$:

$$
\mathbf{c}_t^{KV} = W^{DKV}\,\mathbf{h}_t
$$

dengan $W^{DKV} \in \mathbb{R}^{d_c \times d_{\mathrm{model}}}$ adalah matriks *down-projection*.

Key dan Value untuk seluruh head kemudian direkonstruksi dari vektor laten yang sama melalui *up-projection*:

$$
\mathbf{k}_t^{C} = W^{UK}\,\mathbf{c}_t^{KV}, \qquad
\mathbf{v}_t^{C} = W^{UV}\,\mathbf{c}_t^{KV}
$$

dengan $W^{UK}, W^{UV} \in \mathbb{R}^{(n_h \cdot d_h) \times d_c}$.

Karena $\mathbf{c}_t^{KV}$ berdimensi jauh lebih kecil dibanding $K$ dan $V$ gabungan seluruh head, **hanya $\mathbf{c}_t^{KV}$ yang perlu disimpan di KV cache**, sementara $\mathbf{k}_t^{C}$ dan $\mathbf{v}_t^{C}$ dihitung ulang saat dibutuhkan.

### 4.2 Masalah RoPE pada Key Terkompresi

Positional encoding jenis **RoPE** merotasi vektor $Q$ dan $K$ berdasarkan posisinya sebelum dot-product dihitung (lihat bagian Positional Encoding). Masalahnya, jika RoPE diterapkan langsung pada $\mathbf{k}_t^{C} = W^{UK}\mathbf{c}_t^{KV}$, maka matriks rotasi $R_{pos}$ tidak bisa "dilebur" (*absorbed*) ke dalam $W^{UK}$ karena rotasi bergantung pada posisi $t$, sedangkan $W^{UK}$ bersifat statis. Akibatnya, keuntungan komputasi dari kompresi low-rank hilang — setiap kali attention dihitung, $W^{UK}$ harus dikalikan ulang dengan matriks rotasi yang berbeda-beda per posisi, sehingga proyeksi tidak bisa digabung menjadi satu operasi linear tetap.

### 4.3 Solusi: Decoupled RoPE

MLA mengatasi hal ini dengan memisahkan (*decouple*) komponen yang membawa informasi posisi dari komponen konten. Sebagai tambahan terhadap $\mathbf{k}_t^{C}$, dibuat sebuah **key rotary tambahan berdimensi kecil** $\mathbf{k}_t^{R} \in \mathbb{R}^{d_h^R}$ yang diproyeksikan langsung dari $\mathbf{h}_t$ (bukan dari latent) dan dibagikan ke seluruh head:

$$
\mathbf{k}_t^{R} = \mathrm{RoPE}\big(W^{KR}\,\mathbf{h}_t\big)
$$

Key akhir untuk setiap head adalah hasil konkatenasi antara komponen konten (dari latent) dan komponen posisi (rotary):

$$
\mathbf{k}_t = \big[\,\mathbf{k}_t^{C}\;;\;\mathbf{k}_t^{R}\,\big]
$$

Query dibentuk dengan cara serupa. Untuk efisiensi memori saat training, $Q$ juga dikompresi melalui latent-nya sendiri $\mathbf{c}_t^{Q} = W^{DQ}\mathbf{h}_t$:

$$
\mathbf{q}_t^{C} = W^{UQ}\,\mathbf{c}_t^{Q}, \qquad
\mathbf{q}_t^{R} = \mathrm{RoPE}\big(W^{QR}\,\mathbf{c}_t^{Q}\big)
$$

$$
\mathbf{q}_t = \big[\,\mathbf{q}_t^{C}\;;\;\mathbf{q}_t^{R}\,\big]
$$

Dengan skema ini, hanya komponen $\mathbf{k}_t^{R}$ (yang berdimensi kecil dan dibagikan lintas head) yang membawa rotasi posisi, sedangkan komponen konten $\mathbf{k}_t^{C}$ tetap dapat direkonstruksi secara efisien dari latent tanpa perlu mengalikan ulang dengan matriks rotasi per posisi.

### 4.4 Perhitungan Attention

Skor attention dihitung menggunakan konkatenasi $Q$ dan $K$ di atas:

$$
\mathrm{Attention}(\mathbf{q}_t, \mathbf{k}_{\leq t}, \mathbf{v}_{\leq t}) =
\mathrm{softmax}\left(\frac{\mathbf{q}_t^{\top}\mathbf{k}_{\leq t}}{\sqrt{d_h + d_h^R}}\right)\mathbf{v}_{\leq t}^{C}
$$

dengan:
* $d_h$ adalah dimensi komponen konten per head.
* $d_h^R$ adalah dimensi komponen rotary tambahan (umumnya jauh lebih kecil dari $d_h$).
* Penskalaan $\sqrt{d_h + d_h^R}$ menyesuaikan total dimensi setelah konkatenasi.

### 4.5 Ringkasan Manfaat

| Aspek | MHA Standar | MLA |
|---|---|---|
| Yang disimpan di cache | $K, V$ penuh per head | Vektor laten $\mathbf{c}_t^{KV}$ + $\mathbf{k}_t^{R}$ (berdimensi kecil) |
| Ukuran KV cache | $O(n_h \cdot d_h)$ per token | $O(d_c + d_h^R)$ per token |
| Informasi posisi | Menyatu dengan $K$ | Dipisah (*decoupled*) agar kompresi tetap efisien |

Dengan pendekatan ini, MLA mempertahankan kapasitas representasi yang setara (atau lebih baik) dibanding MHA, sambil secara signifikan mengurangi kebutuhan memori KV cache saat inference — salah satu alasan utama arsitektur ini digunakan pada model skala besar seperti DeepSeek-V2.

In [14]:
class MultiHeadLatentAttention(nn.Module):
    def __init__(
        self,
        d_model: int,       # dimensi embedding utama
        n_heads: int,        # jumlah attention head
        d_head: int,         # dimensi konten per head (non-rotary)
        d_rotary: int,       # dimensi rotary per head (harus genap, karena RoPE bekerja berpasangan)
        d_latent_kv: int,    # dimensi latent untuk Key/Value (d_latent_kv << n_heads * d_head)
        d_latent_q: int,     # dimensi latent untuk Query
        max_seq_len: int = 2048,
        dropout: float = 0.0,
    ):
        super().__init__()
        assert d_rotary % 2 == 0, "d_rotary harus genap (RoPE bekerja per-pasangan dimensi)"

        self.n_heads = n_heads
        self.d_head = d_head
        self.d_rotary = d_rotary
        self.scale = 1.0 / math.sqrt(d_head + d_rotary)

        # --- Jalur Key & Value (dikompresi ke latent bersama) ---
        self.W_DKV = nn.Linear(d_model, d_latent_kv, bias=False)      # down-projection
        self.W_UK = nn.Linear(d_latent_kv, n_heads * d_head, bias=False)  # up-projection -> K konten
        self.W_UV = nn.Linear(d_latent_kv, n_heads * d_head, bias=False)  # up-projection -> V

        # Key rotary: berdimensi kecil, dihitung LANGSUNG dari hidden state (bukan dari latent),
        # dan DIBAGIKAN ke semua head (karena itu tidak diberi indeks head).
        self.W_KR = nn.Linear(d_model, d_rotary, bias=False)

        # --- Jalur Query (juga dikompresi ke latent, untuk efisiensi saat training) ---
        self.W_DQ = nn.Linear(d_model, d_latent_q, bias=False)
        self.W_UQ = nn.Linear(d_latent_q, n_heads * d_head, bias=False)      # Q konten
        self.W_QR = nn.Linear(d_latent_q, n_heads * d_rotary, bias=False)     # Q rotary, per head

        # --- Proyeksi keluaran ---
        self.W_O = nn.Linear(n_heads * d_head, d_model, bias=False)

        # RoPE hanya berdimensi d_rotary, dipakai bersama untuk Q dan K
        self.rope = RotaryPositionEmbedding(d_rotary, max_seq_len)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        """
        x: [batch, seq_len, d_model]
        attn_mask: [seq_len, seq_len] atau None, True/1 = posisi yang DIBOLEHKAN dilihat
        """
        B, T, _ = x.shape

        # ---------- 1. Key & Value dari latent ----------
        c_kv = self.W_DKV(x)                                   # [B, T, d_latent_kv]
        k_C = self.W_UK(c_kv).view(B, T, self.n_heads, self.d_head)  # [B, T, H, d_head]
        v_C = self.W_UV(c_kv).view(B, T, self.n_heads, self.d_head)  # [B, T, H, d_head]

        # Key rotary: satu vektor kecil per token, dibagikan ke seluruh head
        k_R = self.W_KR(x)                 # [B, T, d_rotary]
        k_R = self.rope(k_R)               # rotasi berdasarkan posisi
        k_R = k_R.unsqueeze(2).expand(B, T, self.n_heads, self.d_rotary)  # broadcast ke semua head

        # ---------- 2. Query dari latent ----------
        c_q = self.W_DQ(x)                                     # [B, T, d_latent_q]
        q_C = self.W_UQ(c_q).view(B, T, self.n_heads, self.d_head)   # [B, T, H, d_head]

        q_R = self.W_QR(c_q).view(B, T, self.n_heads, self.d_rotary)  # [B, T, H, d_rotary]
        # Gabungkan (batch, head) jadi satu sumbu supaya bisa dilewatkan ke RoPE (yang menerima [batch, seq, dim])
        q_R = q_R.permute(0, 2, 1, 3).reshape(B * self.n_heads, T, self.d_rotary)
        q_R = self.rope(q_R)
        q_R = q_R.view(B, self.n_heads, T, self.d_rotary).permute(0, 2, 1, 3)  # kembali ke [B, T, H, d_rotary]

        # ---------- 3. Gabungkan komponen konten + rotary ----------
        q = torch.cat([q_C, q_R], dim=-1)   # [B, T, H, d_head + d_rotary]
        k = torch.cat([k_C, k_R], dim=-1)   # [B, T, H, d_head + d_rotary]
        v = v_C                              # Value TIDAK memakai komponen rotary

        # Pindahkan H ke depan agar bisa batched matmul per head: [B, H, T, dim]
        q = q.permute(0, 2, 1, 3)
        k = k.permute(0, 2, 1, 3)
        v = v.permute(0, 2, 1, 3)

        # ---------- 4. Scaled dot-product attention ----------
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # [B, H, T, T]

        if attn_mask is not None:
            scores = scores.masked_fill(~attn_mask.bool(), float("-inf"))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)                     # [B, H, T, d_head]
        out = out.permute(0, 2, 1, 3).reshape(B, T, self.n_heads * self.d_head)

        return self.W_O(out)
    
# Implementasi pada kodenya
d_model = embedded_train.shape[-1]   # sesuaikan dengan dimensi embedding kamu

mla = MultiHeadLatentAttention(
    d_model=d_model,
    n_heads=8,
    d_head=64,
    d_rotary=32,        # umumnya jauh lebih kecil dari d_head
    d_latent_kv=128,    # jauh lebih kecil dari n_heads * d_head (8*64=512)
    d_latent_q=192,
    max_seq_len=2048,
)

# embedded_train diasumsikan berbentuk [batch, seq_len, d_model]
# kalau saat ini bentuknya [seq_len, d_model], tambahkan dimensi batch:
if embedded_train.dim() == 2:
    embedded_train = embedded_train.unsqueeze(0)

output = mla(embedded_train)
print(output.shape)   

torch.Size([1, 512, 512])


## **5. MoE FEED FORWARD (Mixture of Experts)**

**Mixture of Experts (MoE)** adalah modifikasi terhadap lapisan *Feed Forward* standar pada Transformer. Alih-alih menggunakan satu FFN yang sama untuk semua token, MoE menyediakan **beberapa FFN (disebut *expert*)**, namun untuk setiap token hanya **sebagian kecil expert** yang diaktifkan. Pendekatan ini memungkinkan kapasitas parameter model bertambah besar tanpa menambah biaya komputasi secara proporsional, karena mayoritas expert tidak dijalankan pada setiap forward pass.

### Komponen Utama

1. **Router** — menentukan expert mana yang paling relevan untuk suatu token, berdasarkan representasi token itu sendiri.
2. **Routed Experts** — sekumpulan FFN yang dipilih secara dinamis (*top-k*) per token.
3. **Shared Expert** *(opsional, gaya DeepSeek-V2)* — satu atau beberapa FFN yang **selalu aktif** untuk semua token, berfungsi menangkap pengetahuan umum yang dibutuhkan semua token, sekaligus menstabilkan training.

### Perhitungan Routing

Untuk token dengan representasi $\mathbf{h}_t$, router menghitung skor terhadap seluruh expert:

$$
\mathbf{s}_t = \mathrm{softmax}\big(W_r\, \mathbf{h}_t\big)
$$

dengan $W_r \in \mathbb{R}^{N \times d_{\mathrm{model}}}$ dan $N$ adalah jumlah *routed expert*. Hanya $K$ expert dengan skor tertinggi yang diaktifkan ($K \ll N$), lalu bobotnya dinormalisasi ulang agar berjumlah 1:

$$
g_{t,i} = \frac{s_{t,i}}{\sum_{j \in \mathrm{TopK}} s_{t,j}}, \quad i \in \mathrm{TopK}(\mathbf{s}_t, K)
$$

### Output Akhir

Output token $t$ adalah kombinasi dari *shared expert* (selalu ikut) dan *weighted sum* dari *routed expert* terpilih:

$$
\mathbf{y}_t = \sum_{e=1}^{n_{\mathrm{shared}}} \mathrm{FFN}_e^{\mathrm{shared}}(\mathbf{h}_t) \;+\; \sum_{i \in \mathrm{TopK}} g_{t,i}\, \mathrm{FFN}_i(\mathbf{h}_t)
$$

### Masalah Load Balancing

Tanpa kendali tambahan, router cenderung berulang kali memilih expert yang sama (*rich-get-richer*), sehingga expert lain jarang dilatih. Untuk mencegah ini, ditambahkan **auxiliary loss** yang mendorong distribusi token ke seluruh expert menjadi lebih merata:

$$
\mathcal{L}_{\mathrm{aux}} = N \sum_{i=1}^{N} f_i \, P_i
$$

dengan:
* $f_i$ = fraksi token yang benar-benar diarahkan ke expert $i$.
* $P_i$ = rata-rata probabilitas router terhadap expert $i$ di seluruh token.

Loss ini ditambahkan ke loss utama saat training, bukan dipakai saat inference.

In [ ]:
class Expert(nn.Module):
    """
    Satu expert = FFN standar (SwiGLU-style), sama seperti FFN pada
    Transformer biasa, hanya saja nanti akan ada BANYAK expert seperti ini,
    dan hanya sebagian kecil yang aktif per token.
    """

    def __init__(self, d_model: int, d_hidden: int, dropout: float = 0.0):
        super().__init__()
        self.W_gate = nn.Linear(d_model, d_hidden, bias=False)
        self.W_up = nn.Linear(d_model, d_hidden, bias=False)
        self.W_down = nn.Linear(d_hidden, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # SwiGLU: silu(gate) * up, lalu diproyeksikan kembali ke d_model
        h = F.silu(self.W_gate(x)) * self.W_up(x)
        h = self.dropout(h)
        return self.W_down(h)
    
# MoE Feed Forward (Router + Routed Experts + Shared Expert)
class MoEFeedForward(nn.Module):
    """
    Mixture-of-Experts Feed Forward, gaya DeepSeek-V2:

      - n_routed_experts : expert yang dipilih secara dinamis (top-k per token)
      - n_shared_experts  : expert yang SELALU aktif untuk semua token
                             (menstabilkan training, menangkap pengetahuan umum)
      - top_k             : berapa banyak routed expert aktif per token

    Output akhir = kontribusi shared expert + kontribusi weighted dari top-k routed expert.
    """

    def __init__(
        self,
        d_model: int,
        d_hidden: int,
        n_routed_experts: int = 8,
        n_shared_experts: int = 1,
        top_k: int = 2,
        dropout: float = 0.0,
        aux_loss_weight: float = 0.01,
    ):
        super().__init__()
        self.top_k = top_k
        self.n_routed_experts = n_routed_experts
        self.aux_loss_weight = aux_loss_weight

        # --- Router: menentukan expert mana yang relevan untuk tiap token ---
        self.router = nn.Linear(d_model, n_routed_experts, bias=False)

        # --- Kumpulan routed expert (dipilih sebagian per token) ---
        self.routed_experts = nn.ModuleList(
            [Expert(d_model, d_hidden, dropout) for _ in range(n_routed_experts)]
        )

        # --- Shared expert (selalu aktif, tidak melalui routing) ---
        self.shared_experts = nn.ModuleList(
            [Expert(d_model, d_hidden, dropout) for _ in range(n_shared_experts)]
        )

        # Menyimpan aux loss terakhir supaya bisa diambil dari luar saat training
        self.last_aux_loss = None

    def forward(self, x):
        """
        x: [batch, seq_len, d_model]
        return: [batch, seq_len, d_model]
        """
        B, T, D = x.shape
        x_flat = x.view(-1, D)  # [B*T, D] -> routing dilakukan per token, jadi diratakan dulu

        # ---------- 1. Shared expert (selalu aktif untuk semua token) ----------
        shared_out = 0
        for expert in self.shared_experts:
            shared_out = shared_out + expert(x_flat)

        # ---------- 2. Routing: hitung skor tiap token ke tiap routed expert ----------
        router_logits = self.router(x_flat)                  # [B*T, n_routed_experts]
        router_probs = F.softmax(router_logits, dim=-1)       # [B*T, n_routed_experts]

        # Pilih top-k expert dengan probabilitas tertinggi per token
        topk_probs, topk_idx = torch.topk(router_probs, self.top_k, dim=-1)  # [B*T, top_k]
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)       # renormalisasi agar total bobot = 1

        # ---------- 3. Hitung output routed expert HANYA untuk expert yang terpilih ----------
        routed_out = torch.zeros_like(x_flat)

        for expert_id in range(self.n_routed_experts):
            # token_mask: token mana saja yang memilih expert_id ini di salah satu slot top-k
            token_mask, k_slot = (topk_idx == expert_id).nonzero(as_tuple=True)

            if token_mask.numel() == 0:
                continue  # expert ini tidak dipilih oleh token manapun pada batch ini -> skip (efisiensi)

            selected_tokens = x_flat[token_mask]                 # token-token yang memilih expert ini
            expert_output = self.routed_experts[expert_id](selected_tokens)

            # Bobot kontribusi expert ini untuk tiap token yang memilihnya
            weight = topk_probs[token_mask, k_slot].unsqueeze(-1)

            routed_out[token_mask] += expert_output * weight

        # ---------- 4. Gabungkan shared + routed ----------
        out = shared_out + routed_out
        out = out.view(B, T, D)

        # ---------- 5. Auxiliary load-balancing loss ----------
        # Tujuan: mendorong distribusi token ke expert menjadi RATA,
        # supaya tidak ada expert yang "mati" (tidak pernah dipilih) atau
        # segelintir expert yang jadi overload.
        self.last_aux_loss = self._compute_load_balance_loss(router_probs, topk_idx)

        return out

    def _compute_load_balance_loss(self, router_probs, topk_idx):
        """
        f_i = fraksi token yang benar-benar di-routing ke expert i
        P_i = rata-rata probabilitas router terhadap expert i (soft)

        Loss = n_experts * sum(f_i * P_i)
        Loss ini minimal ketika distribusi token ke expert merata.
        (Formulasi mengikuti gaya Switch Transformer / DeepSeek-V2)
        """
        n_experts = self.n_routed_experts
        n_tokens = router_probs.size(0)

        # f_i: fraksi token yang memilih expert i (dihitung dari hasil top-k, bukan soft prob)
        one_hot = F.one_hot(topk_idx, num_classes=n_experts).float()  # [tokens, top_k, n_experts]
        f_i = one_hot.sum(dim=(0, 1)) / (n_tokens * self.top_k)      # [n_experts]

        # P_i: rata-rata probabilitas router (soft) terhadap expert i
        P_i = router_probs.mean(dim=0)                                # [n_experts]

        aux_loss = n_experts * torch.sum(f_i * P_i)
        return self.aux_loss_weight * aux_loss
    
d_model = 512

moe_ffn = MoEFeedForward(
    d_model=d_model,
    d_hidden=1024,        # hidden dim tiap expert, biasanya lebih kecil dari FFN dense biasa
    n_routed_experts=8,   # total expert yang tersedia
    n_shared_experts=1,   # expert yang selalu aktif
    top_k=2,              # tiap token hanya memakai 2 dari 8 routed expert
)

x = torch.randn(4, 16, d_model)  # [batch=4, seq_len=16, d_model=512]
out = moe_ffn(x)

print(out.shape)                 # -> [4, 16, 512]
print(moe_ffn.last_aux_loss)     # -> scalar tensor, WAJIB ditambahkan ke total loss saat training

torch.Size([4, 16, 512])
tensor(0.0102, grad_fn=<MulBackward0>)


In [16]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization.
    Berbeda dari LayerNorm, RMSNorm tidak mengurangi mean (tidak ada centering),
    hanya menskalakan berdasarkan root-mean-square dari elemen vektor.
    """

    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))  # parameter skala yang dipelajari

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        x_normed = x / rms
        return x_normed * self.weight

In [17]:
class TransformerBlock(nn.Module):
    """
    Satu block Transformer bergaya pre-norm:

        h --> RMSNorm --> MLA        --> (+ residual dari h)
          --> RMSNorm --> MoE FFN    --> (+ residual dari hasil sebelumnya)

    PENTING: residual SELALU diambil dari input SEBELUM RMSNorm,
    bukan dari hasil RMSNorm.
    """

    def __init__(self, mla: nn.Module, moe_ffn: nn.Module, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.norm1 = RMSNorm(d_model, eps=eps)
        self.mla = mla

        self.norm2 = RMSNorm(d_model, eps=eps)
        self.moe_ffn = moe_ffn

    def forward(self, x, attn_mask=None):
        # ---------- Sub-layer 1: MLA + residual ----------
        residual = x                          # simpan input ASLI (sebelum normalisasi)
        x_norm = self.norm1(x)
        attn_out = self.mla(x_norm, attn_mask=attn_mask)
        x = residual + attn_out               # residual connection #1

        # ---------- Sub-layer 2: MoE FFN + residual ----------
        residual = x                          # simpan hasil setelah sub-layer 1
        x_norm = self.norm2(x)
        ffn_out = self.moe_ffn(x_norm)
        x = residual + ffn_out                # residual connection #2

        return x

d_model = 512

mla = MultiHeadLatentAttention(
    d_model=d_model,
    n_heads=8,
    d_head=64,
    d_rotary=32,
    d_latent_kv=128,
    d_latent_q=192,
)

moe_ffn = MoEFeedForward(
    d_model=d_model,
    d_hidden=1024,
    n_routed_experts=8,
    n_shared_experts=1,
    top_k=2,
)

block = TransformerBlock(mla=mla, moe_ffn=moe_ffn, d_model=d_model)

x = torch.randn(4, 16, d_model)   # [batch=4, seq_len=16, d_model=512]
out = block(x)

print(out.shape)              # -> [4, 16, 512]
print(moe_ffn.last_aux_loss)   # -> jangan lupa ditambahkan ke total loss saat training

torch.Size([4, 16, 512])
tensor(0.0101, grad_fn=<MulBackward0>)


In [ ]:
class Nvoin_Language_Model(nn.Module):
    """
    Model bahasa lengkap dengan arsitektur:

        Token IDs
          --> Embedding
          --> [TransformerBlock] x N   (tiap block punya MLA & MoE FFN sendiri)
          --> RMSNorm (final)
          --> Linear (proyeksi ke ukuran vocab)
          --> logits (softmax diterapkan di loss function, BUKAN di sini)
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_layers: int,
        n_heads: int,
        d_head: int,
        d_rotary: int,
        d_latent_kv: int,
        d_latent_q: int,
        d_ffn_hidden: int,
        n_routed_experts: int = 8,
        n_shared_experts: int = 1,
        top_k: int = 2,
        max_seq_len: int = 2048,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # ---------- Token Embedding ----------
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        # ---------- Stack N TransformerBlock ----------
        # PENTING: mla dan moe_ffn dibuat BARU di setiap iterasi,
        # agar setiap layer punya parameter sendiri (tidak saling berbagi bobot).
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            mla = MultiHeadLatentAttention(
                d_model=d_model,
                n_heads=n_heads,
                d_head=d_head,
                d_rotary=d_rotary,
                d_latent_kv=d_latent_kv,
                d_latent_q=d_latent_q,
                max_seq_len=max_seq_len,
                dropout=dropout,
            )
            moe_ffn = MoEFeedForward(
                d_model=d_model,
                d_hidden=d_ffn_hidden,
                n_routed_experts=n_routed_experts,
                n_shared_experts=n_shared_experts,
                top_k=top_k,
                dropout=dropout,
            )
            block = TransformerBlock(mla=mla, moe_ffn=moe_ffn, d_model=d_model)
            self.layers.append(block)

        # ---------- Komponen akhir ----------
        self.norm_final = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def _make_causal_mask(self, seq_len: int, device):
        """
        Membuat mask segitiga bawah agar token hanya bisa melihat token
        pada posisi <= dirinya sendiri (autoregressive).
        True  = boleh dilihat
        False = ditutup (di-set -inf saat attention)
        """
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool))
        return mask  # [seq_len, seq_len]

    def forward(self, token_ids: torch.Tensor):
        """
        token_ids: [batch, seq_len]  (berisi ID token, integer)
        return:
            logits    : [batch, seq_len, vocab_size]
            total_aux : scalar tensor, jumlah aux_loss dari SEMUA layer MoE
        """
        B, T = token_ids.shape
        assert T <= self.max_seq_len, f"seq_len ({T}) melebihi max_seq_len ({self.max_seq_len})"

        # ---------- 1. Embedding ----------
        x = self.token_embedding(token_ids)   # [B, T, d_model]
        # Catatan: TIDAK ada penjumlahan positional encoding di sini,
        # karena RoPE sudah diterapkan di dalam MLA pada tiap layer.

        # ---------- 2. Causal mask (dibuat sekali, dipakai semua layer) ----------
        attn_mask = self._make_causal_mask(T, device=token_ids.device)

        # ---------- 3. Lewati semua TransformerBlock ----------
        total_aux_loss = 0.0
        for layer in self.layers:
            x = layer(x, attn_mask=attn_mask)
            total_aux_loss = total_aux_loss + layer.moe_ffn.last_aux_loss

        # ---------- 4. RMSNorm final + output projection ----------
        x = self.norm_final(x)
        logits = self.lm_head(x)   # [B, T, vocab_size]

        return logits, total_aux_loss